In [0]:
# Read from Unity Catalog Bronze Delta table
df_bronze = spark.read.table("fraud_analytics.bronze.credit_card_fraud")

print(f"Bronze row count: {df_bronze.count()}")
df_bronze.printSchema()

In [0]:
from pyspark.sql.functions import (
    col, month, year, lower, regexp_replace, current_timestamp
)

df_silver = df_bronze \
    .withColumnRenamed("TransactionID", "transaction_id") \
    .withColumnRenamed("TransactionDate", "transaction_date") \
    .withColumnRenamed("Amount", "amount") \
    .withColumnRenamed("MerchantID", "merchant_id") \
    .withColumnRenamed("TransactionType", "transaction_type") \
    .withColumnRenamed("Location", "location") \
    .withColumnRenamed("IsFraud", "is_fraud") \
    .withColumnRenamed("ingested_at", "ingested_at") \
    .withColumn("is_fraud", col("is_fraud").cast("boolean")) \
    .withColumn("transaction_month", month(col("transaction_date"))) \
    .withColumn("transaction_year", year(col("transaction_date"))) \
    .withColumn("transformed_at", current_timestamp())

print("Transformations applied successfully")
df_silver.printSchema()

In [0]:
# Record count before quality checks
count_before = df_silver.count()
print(f"Row count before quality checks: {count_before}")

# Drop rows where critical columns are null
df_silver = df_silver.dropna(
    subset=["transaction_id", "amount", "is_fraud"]
)

# Remove duplicates based on transaction_id
df_silver = df_silver.dropDuplicates(["transaction_id"])

# Record count after quality checks
count_after = df_silver.count()
print(f"Row count after quality checks: {count_after}")
print(f"Rows removed: {count_before - count_after}")

In [0]:
df_silver.display()

In [0]:
# Write partitioned Delta table to Unity Catalog silver schema
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("transaction_year", "transaction_month") \
    .saveAsTable("fraud_analytics.silver.credit_card_fraud")

print("Silver table written successfully!")

In [0]:
# Verify silver table
spark.sql("""
    SELECT 
        transaction_year,
        transaction_month,
        COUNT(*) as row_count,
        SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) as fraud_count
    FROM fraud_analytics.silver.credit_card_fraud
    GROUP BY transaction_year, transaction_month
    ORDER BY transaction_year, transaction_month
""").show()

In [0]:
# Check full Delta history
spark.sql("""
    DESCRIBE HISTORY fraud_analytics.silver.credit_card_fraud
""").select("version", "timestamp", "operation").show(10, truncate=False)

In [0]:
# Time travel — query table BEFORE the MERGE (Version 3)
df_before_merge = spark.read \
    .format("delta") \
    .option("versionAsOf", 3) \
    .table("fraud_analytics.silver.credit_card_fraud")

print(f"Row count at Version 3 (before MERGE): {df_before_merge.count()}")

# Current version
df_current = spark.read.table("fraud_analytics.silver.credit_card_fraud")
print(f"Row count at Version 4 (after MERGE): {df_current.count()}")

# Difference
print(f"Rows added by MERGE: {df_current.count() - df_before_merge.count()}")

In [0]:
# OPTIMIZE with Z-ordering
spark.sql("""
    OPTIMIZE fraud_analytics.silver.credit_card_fraud
    ZORDER BY (transaction_id, transaction_date)
""")

# Check how many files before and after
spark.sql("""
    DESCRIBE DETAIL fraud_analytics.silver.credit_card_fraud
""").select("numFiles", "sizeInBytes").show()

# VACUUM — retain 7 days
spark.sql("""
    VACUUM fraud_analytics.silver.credit_card_fraud RETAIN 168 HOURS
""")

print("OPTIMIZE and VACUUM complete!")